In [ ]:
import os
import cv2
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import backend as K

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
!unzip -q BRAIN_TUMOR_SEGMENTATION.zip

In [ ]:
!ls

In [ ]:
mask_paths = glob.glob(
    "/content/kaggle_3m/**/*_mask*",
    recursive=True
)

mask_paths

In [ ]:
image_paths = [
    path.replace("_mask", "")
    for path in mask_paths
]
image_paths

In [ ]:
has_mask = []

for path in tqdm(mask_paths):

    mask = cv2.imread(
        path,
        cv2.IMREAD_GRAYSCALE
    )

    if np.max(mask) > 0:
        has_mask.append(1)

    else:
        has_mask.append(0)

In [ ]:
brain_df = pd.DataFrame()

brain_df["image_path"] = image_paths
brain_df["mask_path"] = mask_paths
brain_df["mask"] = has_mask
brain_df["patient_id"] = brain_df["image_path"].apply(
    lambda x: x.split("/")[-2]
)

In [ ]:
brain_df.to_csv(
    "data_mask.csv",
    index=False
)

In [ ]:
brain_df['mask'].value_counts().plot(kind='bar')

plt.xlabel("Tumor Presence")
plt.ylabel("Count")
plt.title("Tumor vs No Tumor")
plt.show()

In [ ]:
sample = random.randint(
    0,
    len(brain_df)-1
)

image = cv2.imread(
    brain_df.iloc[sample]['image_path']
)

mask = cv2.imread(
    brain_df.iloc[sample]['mask_path'],
    cv2.IMREAD_GRAYSCALE
)

In [ ]:
plt.figure(figsize=(15,5))

plt.subplot(1,2,1)
plt.imshow(image)
plt.title("MRI")

plt.subplot(1,2,2)
plt.imshow(mask, cmap='gray')
plt.title("Mask")

plt.show()

In [ ]:
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(image)
plt.title("MRI")

plt.subplot(1,3,2)
plt.imshow(mask,cmap='gray')
plt.title("Mask")

plt.subplot(1,3,3)
plt.imshow(image)
plt.imshow(mask,cmap='jet',alpha=0.5)
plt.title("Overlay")

plt.show()

In [ ]:
IMG_SIZE = 256

In [ ]:
def load_image(path):

    image = cv2.imread(path)

    image = cv2.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = image / 255.0

    return image.astype(np.float32)

In [ ]:
def load_mask(path):

    mask = cv2.imread(
        path,
        cv2.IMREAD_GRAYSCALE
    )

    mask = cv2.resize(
        mask,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    mask = mask / 255.0

    mask = np.expand_dims(
        mask,
        axis=-1
    )

    return mask.astype(np.float32)

In [ ]:
sample = random.randint(
    0,
    len(brain_df)-1
)

image = load_image(
    brain_df.iloc[sample]["image_path"]
)

mask = load_mask(
    brain_df.iloc[sample]["mask_path"]
)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(image)
plt.title("Processed MRI")

plt.subplot(1,2,2)
plt.imshow(mask.squeeze(), cmap='gray')
plt.title("Processed Mask")

plt.show()

In [ ]:
patients = brain_df["patient_id"].unique()

print("Total patients:", len(patients))
print(patients)

In [ ]:
train_patients, test_patients = train_test_split(
    patients,
    test_size=0.15,
    random_state=42
)

In [ ]:
train_patients, val_patients = train_test_split(
    train_patients,
    test_size=0.15,
    random_state=42
)

In [ ]:
train_df = brain_df[
    brain_df["patient_id"].isin(train_patients)
]

val_df = brain_df[
    brain_df["patient_id"].isin(val_patients)
]

test_df = brain_df[
    brain_df["patient_id"].isin(test_patients)
]


In [ ]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
print(
    set(train_df.patient_id)
    .intersection(set(val_df.patient_id))
)

print(
    set(train_df.patient_id)
    .intersection(set(test_df.patient_id))
)

print(
    set(val_df.patient_id)
    .intersection(set(test_df.patient_id))
)

In [ ]:
BATCH_SIZE = 16

In [ ]:
def data_generator(dataframe, batch_size):

    while True:

        dataframe = dataframe.sample(frac=1)

        for i in range(0, len(dataframe), batch_size):

            batch = dataframe.iloc[i:i+batch_size]

            images = []
            masks = []

            for _, row in batch.iterrows():

                image = load_image(
                    row["image_path"]
                )

                mask = load_mask(
                    row["mask_path"]
                )

                images.append(image)
                masks.append(mask)

            yield (
                np.array(images),
                np.array(masks)
            )

In [ ]:
train_gen = data_generator(
    train_df,
    BATCH_SIZE
)

val_gen = data_generator(
    val_df,
    BATCH_SIZE
)

test_gen = data_generator(
    test_df,
    BATCH_SIZE
)

In [ ]:
images, masks = next(train_gen)

print(images.shape)
print(masks.shape)

In [ ]:
def dice_coef(y_true, y_pred):

    y_true = K.flatten(y_true)
    y_pred = K.flatten(y_pred)

    intersection = K.sum(y_true * y_pred)

    return (2.0 * intersection + 1) / (
        K.sum(y_true) +
        K.sum(y_pred) +
        1
    )

In [ ]:
def iou(y_true, y_pred):

    intersection = K.sum(y_true * y_pred)

    union = (
        K.sum(y_true)
        + K.sum(y_pred)
        - intersection
    )

    return (intersection + 1) / (union + 1)

In [ ]:
def tversky(y_true, y_pred):

    y_true = K.flatten(y_true)
    y_pred = K.flatten(y_pred)

    true_pos = K.sum(y_true * y_pred)

    false_neg = K.sum(y_true * (1 - y_pred))

    false_pos = K.sum((1 - y_true) * y_pred)

    alpha = 0.7

    return (
        true_pos + 1
    ) / (
        true_pos
        + alpha * false_neg
        + (1 - alpha) * false_pos
        + 1
    )

In [ ]:
def focal_tversky_loss(
        y_true,
        y_pred):

    pt_1 = tversky(
        y_true,
        y_pred
    )

    gamma = 0.75

    return K.pow(
        (1 - pt_1),
        gamma
    )

In [ ]:
from tensorflow.keras.layers import *

def residual_block(x, filters):

    x_skip = x

    x = Conv2D(filters, (3,3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2D(filters, (3,3), padding='same')(x)
    x = BatchNormalization()(x)

    x_skip = Conv2D(filters, (1,1), padding='same')(x_skip)

    x = Add()([x, x_skip])
    x = Activation('relu')(x)

    return x

In [ ]:
def encoder_block(x, filters):

    x = residual_block(x, filters)

    p = MaxPool2D((2,2))(x)

    return x, p

In [ ]:
def decoder_block(x, skip, filters):

    x = UpSampling2D((2,2))(x)

    x = Concatenate()([x, skip])

    x = residual_block(x, filters)

    return x

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D

def ResUNet(input_shape=(256,256,3)):

    inputs = Input(input_shape)

    s1, p1 = encoder_block(inputs, 64)

    s2, p2 = encoder_block(p1, 128)

    s3, p3 = encoder_block(p2, 256)

    s4, p4 = encoder_block(p3, 512)


    b1 = residual_block(p4, 1024)


    d1 = decoder_block(b1, s4, 512)

    d2 = decoder_block(d1, s3, 256)

    d3 = decoder_block(d2, s2, 128)

    d4 = decoder_block(d3, s1, 64)

    outputs = Conv2D(
        1,
        (1,1),
        activation='sigmoid'
    )(d4)

    model = Model(inputs, outputs)

    return model

In [ ]:
model = ResUNet()
model.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=focal_tversky_loss,
    metrics=[dice_coef, iou]
)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    ModelCheckpoint(
        "resunet_brain_mri.keras",
        save_best_only=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.1,
        patience=3
    )
]

In [ ]:
print(tf.config.list_physical_devices('GPU'))

images, masks = next(train_gen)
print(images.shape)
print(masks.shape)

print(len(train_df)//BATCH_SIZE)

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    steps_per_epoch=len(train_df)//BATCH_SIZE,
    validation_steps=len(val_df)//BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['dice_coef'], label='Training Dice')
plt.plot(history.history['val_dice_coef'], label='Validation Dice')

plt.xlabel('Epoch')
plt.ylabel('Dice Coefficient')
plt.title('Dice Coefficient Curve')
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['iou'], label='Training IoU')
plt.plot(history.history['val_iou'], label='Validation IoU')

plt.xlabel('Epoch')
plt.ylabel('IoU')
plt.title('IoU Curve')
plt.legend()

plt.show()

In [ ]:
model.evaluate(
    test_gen,
    steps=len(test_df)//BATCH_SIZE
)

In [ ]:
images, masks = next(test_gen)

pred_masks = model.predict(images)

In [ ]:
sample = 0

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(images[sample])
plt.title("MRI")

plt.subplot(1,3,2)
plt.imshow(masks[sample].squeeze(), cmap='gray')
plt.title("Actual Mask")

plt.subplot(1,3,3)
plt.imshow(pred_masks[sample].squeeze(), cmap='gray')
plt.title("Predicted Mask")

plt.show()

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(images[sample])

plt.imshow(
    pred_masks[sample].squeeze(),
    cmap='jet',
    alpha=0.5
)

plt.title("Predicted Tumor Region")

plt.show()

In [ ]:
model.save("resunet_brain_tumor.keras")

In [ ]:
model.save('/content/drive/MyDrive/BRAIN_TUMOR_PROJECT/resunet_brain_tumor.keras')

In [ ]:
import os

os.path.exists(
    '/content/drive/MyDrive/BRAIN_TUMOR_PROJECT/resunet_brain_tumor.keras'
)